# Missão Espacial — treinamento piloto das duas cabeças

Este notebook contém **os dados e o código necessários**. Abra no Colab por **Arquivo → Fazer upload de notebook**, ou em Jupyter na sua máquina, e execute as células na ordem.

A classificação responde **suporte, insuficiente ou contradição**. A outra cabeça responde **probabilidade de entrega no planeta e de retorno com sobrevivência**, dado contexto, objetivo e plano candidato. Quem escolhe a ação é a implementação do agente.

**O treinamento completo não foi executado na preparação deste notebook.** Os resultados aparecerão somente após a execução real. Não há pesos de heads pré-treinados ou números de qualidade pré-preenchidos.

O dispositivo é escolhido automaticamente: **CUDA → Metal/MPS → CPU**. Não exige uma GPU específica. São usados os mesmos pesos Phi-4 de 4 bits em todos os dispositivos. O download inicial tem aproximadamente **8,2 GB**, além do armazenamento e da memória das features/ativações. CPU é permitida, mas pode ser lenta e precisa de memória suficiente. Nenhuma entrada será truncada para caber.

No Colab, GPU é uma opção de aceleração em *Ambiente de execução → Alterar tipo de ambiente de execução*. Este notebook não seleciona um modelo específico de placa.


**Modo padrão: inferência com pesos treinados publicados no GitHub (`main`).** Não precisa repetir as 816 extrações. Para executar o treinamento didático, altere `MODO_EXECUCAO` para `"treinamento"`. A seção 9 avalia 30 casos fora do domínio do jogo usando sempre as cabeças publicadas, mesmo se você treinar outras nesta sessão.


## 1. Dependências

A instalação preserva o PyTorch já disponível se sua versão é compatível, para conservar o suporte do ambiente à aceleração. O treino das heads roda na CPU; a extração das features usa o dispositivo escolhido. Para execução local por terminal, o pacote exportado contém `requirements.txt` e os comandos no README.


In [ ]:
import os, sys, subprocess, importlib.util
VALIDACAO_LEVE = os.environ.get("MISSAO_VALIDACAO_LEVE") == "1"
if not VALIDACAO_LEVE:
    requirements = ["transformers==4.57.6", "numpy>=1.26,<3", "safetensors==0.8.0", "huggingface-hub==0.36.2", "certifi>=2024.2.2"]
    if importlib.util.find_spec("torch") is None:
        requirements.append("torch>=2.5,<3")
    else:
        import torch
        version = tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2])
        if version < (2, 5):
            raise RuntimeError("Atualize o PyTorch para versão >= 2.5 no seu ambiente antes de continuar.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])
else:
    print("Validação leve: dependências existentes; sem download ou treino Phi-4.")


## 2. Configuração e persistência

Para retomar entre sessões do Colab, monte seu Drive e ajuste `WORK_DIR` para uma pasta persistente, por exemplo `/content/drive/MyDrive/missao-treino`. A montagem do Drive fica a seu critério; o notebook não acessa sua conta automaticamente. `HF_CACHE_DIR` pode apontar para uma pasta persistente de pesos.

`MODEL_DIRECTORY = None` baixa os pesos públicos fixos. Se já os tiver, informe a pasta — os arquivos serão conferidos. `DEVICE="auto"` é o padrão; `"cpu"` força CPU. `PREFILL_CHUNK=None` usa 256 tokens por bloco na CPU e prompt inteiro em aceleradores. Se faltar memória, use `PREFILL_CHUNK=256` em qualquer dispositivo: **o cache conserva todos os tokens anteriores**. Isso altera o custo de execução, não a janela. Não reutilize o mesmo cache de features ao mudar hardware, versão do runtime ou essa configuração; use outro `WORK_DIR`.


`MODO_EXECUCAO="inferencia-github"` pula as etapas de extração e treinamento e executa a avaliação fora do domínio. `"treinamento"` conserva o percurso original e também permite comparar a avaliação das cabeças publicadas. `EXECUTAR_OOD=False` permite desativar essa avaliação. O corpo Phi-4 continua necessário à inferência: os arquivos pequenos no GitHub são as cabeças, não o corpo de 8,2 GB.


In [ ]:
from pathlib import Path
WORK_DIR = Path(os.environ.get("MISSAO_WORK_DIR", "missao-espacial-treino"))
HF_CACHE_DIR = WORK_DIR / "cache-pesos"
MODEL_DIRECTORY = None
DEVICE = "auto"
TILE_ROWS = 256
PREFILL_CHUNK = None
MODO_EXECUCAO = "inferencia-github"  # ou "treinamento"
EXECUTAR_OOD = not VALIDACAO_LEVE
if MODO_EXECUCAO not in ("inferencia-github", "treinamento"):
    raise ValueError("MODO_EXECUCAO inválido")
EXECUTAR_TREINAMENTO = not VALIDACAO_LEVE and MODO_EXECUCAO == "treinamento"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR.resolve())
print("Trabalho:", WORK_DIR.resolve())
print("Modo:", MODO_EXECUCAO)
print("Executar treinamento:", EXECUTAR_TREINAMENTO)
print("Executar inferência fora do domínio:", EXECUTAR_OOD)


## 3. Abrir o pacote incorporado

O pacote contém 600 pares NLI e todos os 216 exemplos de desfechos do v04. As entradas e a supervisão estão em arquivos diferentes. Nenhum registro de roteamento, ID real de participante, testemunha ou justificativa é enviado ao modelo. O bloco de dados é comprimido e sua integridade será conferida.


In [ ]:
import base64, hashlib, io, zipfile, json
PAYLOAD = "..."
EXPECTED_SHA256 = "0e7714fb0b6c29c762c254418cb4e93fd0d3aa57685b932078ff537b1975e13c"
raw = base64.b64decode(PAYLOAD)
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Pacote incorporado corrompido")
PACKAGE_DIR = WORK_DIR / "pacote"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for info in archive.infolist():
        target = (PACKAGE_DIR / info.filename).resolve()
        if not target.is_relative_to(PACKAGE_DIR.resolve()) or info.is_dir():
            raise RuntimeError("Caminho inválido no pacote")
        content = archive.read(info)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError(f"Arquivo existente diverge do pacote: {target}. Use outro WORK_DIR.")
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(content)
sys.path.insert(0, str(PACKAGE_DIR / "ferramentas"))
from treinar import read_data
inputs, supervision = read_data(PACKAGE_DIR)
print("Pacote íntegro:", len(inputs), "exemplos")


## 4. Conferir o protocolo e executar os testes

NLI: 420 treino, 90 calibração, 90 teste — 200 exemplos de cada classe no total. Episódios, contextos e variantes relacionadas permanecem na mesma partição.

Desfechos: 144 treino, 36 calibração, 36 teste. As famílias estruturais do v04 permanecem separadas. Cada alvo foi estimado com 32 continuações pareadas da engine. As 6.912 continuações **não** são 6.912 exemplos independentes de treino.

A normalização usa apenas treino. São duas heads lineares, 400 épocas fixas, L2 0,1, Adam e learning rate 0,01. A temperatura e o limiar demonstrativo são ajustados na calibração; o teste não participa dessas escolhas.

Os testes usam operações CPU, um Phi-3 minúsculo aleatório e features sintéticas. Validam execução e contratos, **não medem a qualidade do Phi-4**.


In [ ]:
protocol = json.loads((PACKAGE_DIR / "protocolo.json").read_text())
print(json.dumps(protocol, ensure_ascii=False, indent=2))
subprocess.check_call([sys.executable, "-B", str(PACKAGE_DIR / "ferramentas" / "testar.py")])


## 5. Extrair features reais do Phi-4

Esta é a etapa mais cara. O corpo do modelo fica congelado; a feature é o último estado oculto do prompt completo (5.120 dimensões). O prompt da tarefa contém apenas a entrada e as regras constantes. A outra saída desta etapa é a projeção A/B/C do modelo, usada somente como baseline NLI.

Um checkpoint é salvo a cada exemplo. Executar novamente retoma as linhas restantes depois de conferir integridade, ambiente e identidade do modelo. Mantenha o diretório completo. O notebook recusa treinamento se faltarem features. **Não há resultado de qualidade enquanto esta etapa e o treino não terminarem.**


In [ ]:
from treinar import extract
if EXECUTAR_TREINAMENTO:
    extraction = extract(PACKAGE_DIR, WORK_DIR, MODEL_DIRECTORY, DEVICE, TILE_ROWS, PREFILL_CHUNK)
    print("Features concluídas:", len(extraction["rows"]))
else:
    print("Extração de treino pulada neste modo.")


## 6. Treinar, calibrar e avaliar

As heads são treinadas na CPU e exportadas com normalização, temperatura e contrato. O relatório mostra as versões crua e calibrada separadamente.

NLI: acurácia, macro F1, matriz de confusão, perda e métricas por classe; comparação com distribuição uniforme e projeção zero-shot A/B/C do Phi-4 (não é a cabeça do Crivo).

Desfechos: Brier e perda Bernoulli calculados sobre as frequências da engine, bins de calibração e comparação com a frequência média do treino. O limiar usa custos unitários de exemplo; ele não determina a intenção do agente. O diagnóstico de planos compara apenas três candidatos e não prova uma estratégia ótima ou generalização universal.


In [ ]:
from treinar import train
if EXECUTAR_TREINAMENTO:
    result = train(PACKAGE_DIR, WORK_DIR)
    nli_test = result["tasks"]["nli"]["evaluation"]["test"]
    forecast_test = result["tasks"]["forecast"]["evaluation"]
    print("NLI — teste:", json.dumps(nli_test, ensure_ascii=False, indent=2))
    for name in ("raw_head", "calibrated_head", "train_prior_baseline"):
        print("Desfechos — teste —", name)
        print(json.dumps(forecast_test[name]["partitions"]["test"], ensure_ascii=False, indent=2))
    print("Relatório completo:", WORK_DIR / "resultado.json")
else:
    print("Treinamento pulado; a seção 9 pode usar as cabeças já treinadas da main.")


## 7. Inferência experimental

A saída NLI é apenas a classificação. A outra saída são as duas probabilidades dos desfechos, cuja soma é um. A implementação mantém o vínculo com o destinatário, compara alternativas e decide a ação.

A célula abaixo usa um exemplo reservado para demonstração de inferência, sem refazer treinamento ou calibração. A cabeça probabilística permanece condicionada ao amostrador do v04 e aos três programas candidatos. Estes artefatos não substituem automaticamente as heads do simulador ou do Crivo.


In [ ]:
if EXECUTAR_TREINAMENTO:
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    from inferir import PilotPredictor
    predictor = PilotPredictor(PACKAGE_DIR, WORK_DIR / "heads", MODEL_DIRECTORY, DEVICE, TILE_ROWS, PREFILL_CHUNK)
    for task in ("nli", "forecast"):
        idx = next(r["index"] for r in supervision if r["task"] == task and r["split"] == "test")
        print(task, predictor.predict(task, inputs[idx]))
    del predictor
else:
    print("Demonstração com cabeças desta sessão pulada; siga para a seção 9.")


## 8. Exportar e interpretar o resultado

O pacote exportado inclui os dados, scripts, contrato do modelo e instruções para rodar localmente. Após o treino real, também inclui heads, previsões reservadas e relatório. Os pesos do corpo Phi-4 e o cache de features não entram no ZIP final. Para retomar a extração entre sessões, preserve o diretório de features separadamente no armazenamento persistente.

Se as heads não superarem os baselines, registre isso e amplie/revise a massa antes de integrar. Este piloto não demonstra cobertura de todos os contextos, perguntas ou estratégias. As heads nunca escolhem a ação do agente.


In [ ]:
export_path = WORK_DIR / "missao-espacial-piloto-exportacao.zip"
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in PACKAGE_DIR.rglob("*"):
        if item.is_file() and "__pycache__" not in item.parts:
            archive.write(item, "pacote/" + str(item.relative_to(PACKAGE_DIR)))
    if EXECUTAR_TREINAMENTO:
        for item in (WORK_DIR / "heads").rglob("*"):
            if item.is_file():
                archive.write(item, "heads/" + str(item.relative_to(WORK_DIR / "heads")))
        archive.write(WORK_DIR / "resultado.json", "resultado.json")
print("Exportação:", export_path.resolve())
print("Contém heads treinadas:", EXECUTAR_TREINAMENTO)
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if not VALIDACAO_LEVE and IN_COLAB:
    from google.colab import files
    files.download(str(export_path))


## 9. Experimento: a mesma cabeça consegue classificar outros assuntos?

O treinamento utilizou textos do jogo. Agora mantemos os pesos e o processo de inferência e mudamos o assunto dos textos: papelaria, arquivo, alarme, peças e amostras de materiais.

**O que o aluno fará:** ler uma entrada completa, prever sua classe, observar como a entrada é preparada para o Phi-4, executar a cabeça e comparar a classificação com o gabarito.

A tarefa continua a mesma: **a hipótese é sustentada, contradita ou não resolvida pela premissa?** Não estamos perguntando qual ação tomar nem pedindo ao modelo conhecimento externo sobre esses assuntos.

### 9.1. O que é uma entrada?

Um exemplo é um objeto JSON com exatamente dois campos de texto:

| Campo | Pergunta que ele responde | Conteúdo |
|---|---|---|
| `premise` — premissa | O que está disponível para decidir? | Contexto, momento, fatos confirmados, relatos, suposições e regras explícitas. |
| `hypothesis` — hipótese | Qual afirmação será avaliada? | Uma declaração específica sobre esse contexto. |

A hipótese é uma **afirmação**, não precisa ser uma pergunta. Para investigar “há estoque suficiente?”, podemos avaliar a afirmação “um pedido de 14 envelopes pode ser atendido”. Cada hipótese forma um exemplo separado, mesmo quando a premissa é a mesma.

**Os subtítulos dentro da premissa são texto.** O classificador não recebe seis campos estruturados nem uma lista de fatos extraída por outro sistema. Ele recebe as duas strings completas. Os títulos ajudam a distinguir o estatuto de cada informação.

| Rótulo | Relação com a premissa | Cuidado |
|---|---|---|
| suporte | A evidência e as regras sustentam a hipótese. | Não basta parecer plausível. |
| contradição | A evidência e as regras são incompatíveis com a hipótese. | Ausência de confirmação não é negação. |
| insuficiente | A premissa não resolve a hipótese nem sua negação. | Um relato pode provar o que foi dito sem provar o fato relatado. |

A saída pública tem um único campo: `{"classification": "suporte"}`, `{"classification": "insuficiente"}` ou `{"classification": "contradição"}`.


### 9.2. Exemplo guiado: uma premissa, três hipóteses

Esta é a **entrada completa e real** do caso 16 (índice 15 do arquivo). Os caracteres `\n` no JSON representam quebras de linha do texto da premissa.

```json
{
  "premise": "CONTEXTO DISPONÍVEL\nEstoque de papelaria, às 10h. Todas as declarações de estoque se referem a esse instante.\n\nFATOS CONFIRMADOS\nO estoque contém exatamente 14 envelopes. Os envelopes estão lacrados. O pedido atual existe, mas sua quantidade não foi informada.\n\nRELATOS REGISTRADOS — NÃO CONFIRMAM O FATO RELATADO\nNenhum relato adicional.\n\nSUPOSIÇÕES — NÃO SÃO CONFIRMAÇÕES\nNenhuma suposição adicional.\n\nREGRAS EXPLÍCITAS\nUm pedido de envelopes pode ser atendido integralmente se, e somente se, a quantidade pedida for menor ou igual ao estoque disponível.",
  "hypothesis": "Um pedido de 14 envelopes pode ser atendido integralmente às 10h."
}
```

Para decidir, identificamos **14 envelopes confirmados**, a regra **quantidade pedida ≤ estoque** e a hipótese **pedido de 14**. Como **14 ≤ 14**, a classe esperada é **suporte**.

Mantendo exatamente a mesma premissa, os arquivos contêm três exemplos:

| Caso no notebook | Hipótese enviada | Raciocínio para o gabarito | Classe esperada |
|---|---|---|---|
| 16 | Um pedido de 14 envelopes pode ser atendido integralmente às 10h. | 14 ≤ 14: a regra permite atender. | suporte |
| 19 | Um pedido de 15 envelopes pode ser atendido integralmente às 10h. | 15 > 14: a regra impede atender. | contradição |
| 17 | O pedido atual pode ser atendido integralmente às 10h. | O pedido existe, mas sua quantidade é desconhecida. Não podemos aplicar a comparação. | insuficiente |

**Distinção essencial:** a premissa não mudou; mudou a afirmação avaliada. Um contexto rico pode continuar insuficiente para uma hipótese específica. O gabarito acima é uma explicação para o aluno e **não entra no modelo**.


### 9.3. Exemplo contrastante: “foi dito” versus “é fato”

Leia a premissa completa desta segunda família:

```text
CONTEXTO DISPONÍVEL
Análise de uma mensagem sobre um alarme, às 9h.

FATOS CONFIRMADOS
A mensagem abaixo foi registrada integralmente às 9h. A porta da sala está confirmadamente fechada.

RELATOS REGISTRADOS — NÃO CONFIRMAM O FATO RELATADO
Há um único relato registrado: «O alarme está acionado às 9h». Não há medição confirmada do estado do alarme.

SUPOSIÇÕES — NÃO SÃO CONFIRMAÇÕES
Nenhuma suposição adicional.

REGRAS EXPLÍCITAS
Nenhuma regra adicional.
```

| Caso | Hipótese enviada | Classe esperada | Por quê? |
|---|---|---|---|
| 12 | O relato registrado afirma que o alarme está acionado às 9h. | suporte | A hipótese descreve o conteúdo registrado, não o estado real do alarme. |
| 14 | O alarme está realmente acionado às 9h. | insuficiente | O registro comprova o relato; não confirma seu conteúdo como fato. |
| 25 | O único relato registrado afirma que o alarme está desativado às 9h. | contradição | O conteúdo integral do único relato é acionado, e não desativado. |

A mensagem registrada confirma a existência e o conteúdo do relato. Não há medição do alarme. Por isso, **o relato foi registrado** pode receber suporte, enquanto **o alarme está realmente acionado** recebe insuficiente. Não é uma regra “todo relato = insuficiente”: depende do que a hipótese afirma.

**Preveja antes de executar:** se uma medição confirmada às 9h dissesse que o alarme está desativado, qual seria a classe da hipótese sobre o estado real? Explique qual evidência mudou a decisão.


### 9.4. Da entrada do caso à entrada efetiva do Phi-4

Há duas etapas diferentes. O notebook fornece o par; o carregador prepara o prompt exatamente como no treinamento:

```text
Par do caso: {premise, hypothesis}
                 ↓
Premissa efetiva = prefixo fixo + duas quebras de linha + premise
Hipótese efetiva = hypothesis, sem alteração
                 ↓
Mensagens: instrução system + JSON do par efetivo na mensagem user
                 ↓
Chat template do Phi-4 + tokenização, sem truncamento
                 ↓
Phi-4 congelado → representação de 5.120 valores
                 ↓
Normalização e cabeça treinadas → classification
```

**O prefixo também é parte da entrada do modelo.** Ele contém regras sobre evidência e regras específicas do jogo. Mantemos todas porque a cabeça foi treinada com esse prefixo. Portanto, o experimento avalia novos assuntos **sob o protocolo original**, não um classificador sem qualquer referência ao jogo.

A instrução `system` original usa A/B/C. Ela foi utilizada para extrair as representações durante o treinamento e permanece por compatibilidade. Nesta inferência, **não pedimos ao modelo que gere uma letra**: a cabeça aplica seus pesos à representação e retorna uma das três classes.

A célula seguinte mostra a instrução e o prefixo completos, antes de carregar o corpo Phi-4. Mais adiante, veremos o prompt serializado e sua quantidade de tokens para um caso escolhido.


In [ ]:
from phi4_portatil import NLI_SYSTEM
PREFIXO_OOD = (PACKAGE_DIR / "dados/prefixo-nli.txt").read_text().rstrip("\n")
print("INSTRUÇÃO SYSTEM ORIGINAL:\n" + NLI_SYSTEM)
print("\nPREFIXO FIXO COMPLETO, ACRESCENTADO À PREMISSA:\n" + PREFIXO_OOD)


### 9.5. Como estão separados os 30 casos?

| Família | Base | Variantes | Total | O que exercita |
|---|---:|---:|---:|---|
| Estoque | 3 | 1 | 4 | Regra explícita e quantidade do pedido desconhecida. |
| Fronteira numérica | 3 | 1 | 4 | `> 14`, `≥ 14` e quantidade futura desconhecida. |
| Tempo | 3 | 1 | 4 | Leitura no momento correto e ausência de previsão futura. |
| Relato | 3 | 1 | 4 | Conteúdo registrado versus estado real do ambiente. |
| Relatos conflitantes | 3 | 0 | 3 | Divergência entre relatos não resolve o fato. |
| Suposição | 3 | 1 | 4 | Separar hipótese não verificada de observação confirmada. |
| Categorias abertas | 3 | 1 | 4 | A presença de cobre não exclui a presença de ferro. |
| Regra explícita | 3 | 0 | 3 | Aplicar condições declaradas; não inventar evidência de assinatura. |
| **Total** | **24** | **6** | **30** | **10 por classe.** |

Os seis exemplos variantes modificam redação e acrescentam informação irrelevante. São relacionados aos exemplos de base; não equivalem a seis observações independentes novas. Os rótulos foram definidos antes da inferência. Esta amostra não cobre todos os assuntos ou formas de escrever.

| Arquivo | Uso | Entra no prompt? |
|---|---|---|
| `inputs.jsonl` | Os 30 objetos com premissa e hipótese. | Sim, um objeto por inferência, preparado com o prefixo. |
| `supervisao.jsonl` | Classe esperada, justificativa, família, índice e variante. | **Não.** Serve para comparar e interpretar depois. |
| `protocolo.json` | Distribuição, limites do experimento e hashes dos dados. | **Não.** Serve para validar a avaliação. |
| `nli.npz` e `artefato.json` | Pesos, normalização, temperatura e contrato da cabeça. | Não são texto do prompt; configuram a inferência. |

O índice do caso organiza a avaliação. Não é uma identidade de agente e não é passado ao modelo. A cabeça de desfechos não é testada fora da missão: entrega e retorno têm significados específicos do jogo.

**Origem dos arquivos:** resolvemos o commit atual da `main` e baixamos pesos e casos desse mesmo commit. Os hashes são conferidos. O corpo Phi-4 é obtido da revisão fixa do pacote; as cabeças já treinadas vêm do GitHub. Nenhum treino ou ajuste de temperatura ocorre aqui.


In [ ]:
import urllib.request, re, ssl, certifi
GITHUB_REPO = "AIConnection/missao-espacial"
OOD_DIR = WORK_DIR / "avaliacao-fora-dominio-github"
GITHUB_HEADS_DIR = None
OOD_INPUTS, OOD_SUPERVISION = [], []
GITHUB_COMMIT = None

def baixar_github(url):
    request = urllib.request.Request(url, headers={"User-Agent": "missao-espacial-aula05"})
    with urllib.request.urlopen(request, timeout=120, context=ssl.create_default_context(cafile=certifi.where())) as response:
        return response.read()

if EXECUTAR_OOD:
    ref = json.loads(baixar_github(f"https://api.github.com/repos/{GITHUB_REPO}/git/ref/heads/main"))
    GITHUB_COMMIT = ref["object"]["sha"]
    if not re.fullmatch(r"[0-9a-f]{40}", GITHUB_COMMIT):
        raise ValueError("Commit remoto inválido")
    raw_url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_COMMIT}/"
    version_dir = OOD_DIR / GITHUB_COMMIT
    GITHUB_HEADS_DIR = version_dir / "heads"
    GITHUB_HEADS_DIR.mkdir(parents=True, exist_ok=True)
    head_path = "treinamento/piloto-v01/execucao-local-01/heads/"
    artifact_bytes = baixar_github(raw_url + head_path + "artefato.json")
    artifact = json.loads(artifact_bytes)
    if artifact["version"] != "mission-pilot-heads-v1":
        raise ValueError("Versão de cabeça incompatível")
    if artifact["package_sha256"] != hashlib.sha256((PACKAGE_DIR / "integridade.json").read_bytes()).hexdigest():
        raise ValueError("O pacote deste notebook difere do treinamento publicado. Atualize o notebook.")
    for name, expected in artifact["code_sha256"].items():
        if hashlib.sha256((PACKAGE_DIR / "ferramentas" / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Código incompatível: " + name)
    for task in ("nli", "forecast"):
        content = baixar_github(raw_url + head_path + task + ".npz")
        if hashlib.sha256(content).hexdigest() != artifact["heads"][task]:
            raise ValueError("Peso corrompido: " + task)
        (GITHUB_HEADS_DIR / (task + ".npz")).write_bytes(content)
    (GITHUB_HEADS_DIR / "artefato.json").write_bytes(artifact_bytes)
    case_path = "treinamento/piloto-v01/avaliacao-fora-dominio-01/"
    ood_protocol_bytes = baixar_github(raw_url + case_path + "protocolo.json")
    ood_protocol = json.loads(ood_protocol_bytes)
    (version_dir / "protocolo.json").write_bytes(ood_protocol_bytes)
    for name in ("inputs.jsonl", "supervisao.jsonl"):
        content = baixar_github(raw_url + case_path + name)
        if hashlib.sha256(content).hexdigest() != ood_protocol["data_sha256"][name]:
            raise ValueError("Casos corrompidos: " + name)
        (version_dir / name).write_bytes(content)
    OOD_INPUTS = [json.loads(line) for line in (version_dir / "inputs.jsonl").read_text().splitlines()]
    OOD_SUPERVISION = [json.loads(line) for line in (version_dir / "supervisao.jsonl").read_text().splitlines()]
    if len(OOD_INPUTS) != 30 or len(OOD_SUPERVISION) != 30:
        raise ValueError("A avaliação requer 30 pares")
    for index, (value, target) in enumerate(zip(OOD_INPUTS, OOD_SUPERVISION)):
        if set(value) != {"premise", "hypothesis"} or not all(isinstance(v, str) for v in value.values()):
            raise ValueError("Entrada fora do contrato")
        if target["index"] != index or target["expected"] not in ("suporte", "insuficiente", "contradição"):
            raise ValueError("Supervisão desalinhada")
    from collections import Counter
    if Counter(t["expected"] for t in OOD_SUPERVISION) != {"suporte": 10, "insuficiente": 10, "contradição": 10}:
        raise ValueError("Distribuição diferente do protocolo")
    print("Commit da main:", GITHUB_COMMIT)
    print("Pesos conferidos:", artifact["heads"])
    print("Casos por classe:", dict(Counter(t["expected"] for t in OOD_SUPERVISION)))
else:
    print("Downloads OOD desativados.")


### 9.6. Inspecione um caso e formule sua previsão

Escolha `CASO_PARA_INSPECIONAR` entre **1 e 30**. A célula mostra exatamente o par enviado ao carregador. Primeiro leia a premissa e a hipótese e escreva sua classe e justificativa. Depois altere `MOSTRAR_GABARITO=True` para conferir a resposta esperada.

Use `MOSTRAR_TODOS_OS_CASOS=True` somente se quiser examinar a massa inteira. O gabarito fica sempre separado da entrada.


In [ ]:
CASO_PARA_INSPECIONAR = 14  # numeração didática: 1 a 30; índice interno = caso - 1
MOSTRAR_GABARITO = False
MOSTRAR_TODOS_OS_CASOS = False
if EXECUTAR_OOD:
    if not 1 <= CASO_PARA_INSPECIONAR <= len(OOD_INPUTS):
        raise ValueError("Escolha um caso entre 1 e 30")
    indices = range(len(OOD_INPUTS)) if MOSTRAR_TODOS_OS_CASOS else [CASO_PARA_INSPECIONAR - 1]
    for index in indices:
        value, target = OOD_INPUTS[index], OOD_SUPERVISION[index]
        print(f"\nCASO {index + 1:02d} — {target['family']} — {target['variant']} (metadados de avaliação)")
        print("PAR COMPLETO ENVIADO AO CARREGADOR:")
        print(json.dumps(value, ensure_ascii=False, indent=2))
        print("\nPREMISSA EM TEXTO LEGÍVEL:\n" + value["premise"])
        print("\nHIPÓTESE:\n" + value["hypothesis"])
        if MOSTRAR_GABARITO:
            print("\nGABARITO SEPARADO — NÃO É INPUT DO MODELO:")
            print("CLASSE: " + target["expected"])
            print("JUSTIFICATIVA: " + target["rationale"])


### 9.7. Executar inferência real

São 30 forwards do Phi-4 congelado e da cabeça publicada. O dispositivo é selecionado automaticamente; CPU pode demorar. O primeiro uso baixa cerca de 8,2 GB do corpo Phi-4 na revisão compatível. A janela é a original, sem truncamento.

Os resultados de **cada nova execução** ficam em uma pasta própria. A célula não carrega as previsões históricas para fingir inferência. A normalização, os pesos e a temperatura da cabeça permanecem intactos. Não há escolha de limiar para modificar as três classes.


In [ ]:
OOD_RESULTS = []
OOD_RUN_DIR = None
if EXECUTAR_OOD:
    import gc, time, tempfile, torch
    from inferir import PilotPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    predictor_ood = PilotPredictor(PACKAGE_DIR, GITHUB_HEADS_DIR, MODEL_DIRECTORY, DEVICE, TILE_ROWS, PREFILL_CHUNK)
    try:
        index_exemplo = CASO_PARA_INSPECIONAR - 1
        par_exemplo = OOD_INPUTS[index_exemplo]
        par_efetivo = {"premise": predictor_ood.prefix + "\n\n" + par_exemplo["premise"],
                       "hypothesis": par_exemplo["hypothesis"]}
        mensagens = [{"role": "system", "content": NLI_SYSTEM},
                     {"role": "user", "content": json.dumps(par_efetivo, ensure_ascii=False)}]
        prompt_exemplo = predictor_ood.backend.tokenizer.apply_chat_template(
            mensagens, tokenize=False, add_generation_prompt=True)
        tokens_exemplo = predictor_ood.backend.encode_nli(par_exemplo, predictor_ood.prefix)
        assert tokens_exemplo == predictor_ood.backend.tokenizer.encode(prompt_exemplo, add_special_tokens=False)
        print(f"PROMPT COMPLETO DO CASO {CASO_PARA_INSPECIONAR:02d} APÓS O CHAT TEMPLATE:\n{prompt_exemplo}")
        print("Quantidade de tokens:", len(tokens_exemplo), "| Janela:", predictor_ood.backend.window)
        print("Apenas a entrada e o protocolo fixo estão acima; o gabarito ficou fora.")
        OOD_RUN_DIR = Path(tempfile.mkdtemp(prefix="execucao-", dir=GITHUB_HEADS_DIR.parent))
        for index, (value, target) in enumerate(zip(OOD_INPUTS, OOD_SUPERVISION)):
            start = time.perf_counter()
            output = predictor_ood.predict("nli", value)
            if set(output) != {"classification"} or output["classification"] not in ("suporte", "insuficiente", "contradição"):
                raise ValueError("Saída fora do contrato")
            row = {"index": index, "family": target["family"], "expected": target["expected"],
                   "output": output, "correct": output["classification"] == target["expected"],
                   "seconds": time.perf_counter() - start}
            OOD_RESULTS.append(row)
            with (OOD_RUN_DIR / "previsoes.jsonl").open("a", encoding="utf-8") as stream:
                stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
            print(f"{index + 1:02d}/30 | esperado={row['expected']} | previsto={output['classification']} | {row['seconds']:.1f}s", flush=True)
    finally:
        del predictor_ood
        gc.collect()
else:
    print("Inferência fora do domínio desativada.")


### 9.8. Avaliar e interpretar

**Acurácia** mede a fração de acertos. **Precisão** mede quantas previsões de uma classe estão corretas; **recall** mede quantos exemplos dessa classe foram reconhecidos. **Macro F1** dá o mesmo peso às três classes. A matriz de confusão mostra quais classes foram confundidas: linhas são os rótulos esperados, colunas são as previsões.

Observe especialmente insuficiente. Confira erros lendo premissa e hipótese, distinguindo fato, relato e suposição. O teste não define uma estratégia para o agente.

A execução local anterior acertou 23/30, com sete erros em insuficiente. Isso é uma referência histórica, **não uma saída garantida** para todo dispositivo. Não copiamos essas previsões para a avaliação atual. O teste é exploratório, com assuntos e formulações novos; não isola a causa dos erros. Duas variantes mudam paráfrase e evidência irrelevante juntas. Não se pode atribuir o efeito a apenas um desses fatores.

Perguntas para discussão: qual evidência seria necessária para mudar insuficiente para suporte? A contradição exige uma negação comprovada? O acerto no teste reservado do treinamento demonstra generalização para qualquer domínio?


In [ ]:
if EXECUTAR_OOD:
    import numpy as np
    labels_ood = ["suporte", "insuficiente", "contradição"]
    if len(OOD_RESULTS) != len(OOD_INPUTS):
        raise RuntimeError("Execução parcial: conclua os 30 casos antes de calcular as métricas.")
    confusion = np.zeros((3, 3), dtype=int)
    for row in OOD_RESULTS:
        confusion[labels_ood.index(row["expected"]), labels_ood.index(row["output"]["classification"])] += 1
    per_class = {}
    for i, label in enumerate(labels_ood):
        tp = int(confusion[i, i])
        predicted = int(confusion[:, i].sum())
        actual = int(confusion[i, :].sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / actual if actual else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[label] = {"precision": precision, "recall": recall, "f1": f1, "count": actual}
    ood_result = {"github_commit": GITHUB_COMMIT, "heads_sha256": artifact["heads"],
                  "rows": len(OOD_RESULTS), "device": DEVICE,
                  "accuracy": float(np.trace(confusion) / confusion.sum()),
                  "macro_f1": sum(v["f1"] for v in per_class.values()) / 3,
                  "labels": labels_ood, "confusion": confusion.tolist(), "per_class": per_class,
                  "seconds": sum(r["seconds"] for r in OOD_RESULTS)}
    print(json.dumps(ood_result, ensure_ascii=False, indent=2, allow_nan=False))
    errors = [r for r in OOD_RESULTS if not r["correct"]]
    print("\nErros:", len(errors))
    for row in errors:
        i = row["index"]
        print(f"\nCaso {i + 1:02d}: esperado={row['expected']}; previsto={row['output']['classification']}")
        print("PREMISSA:\n" + OOD_INPUTS[i]["premise"])
        print("HIPÓTESE: " + OOD_INPUTS[i]["hypothesis"])
        print("RATIONALE: " + OOD_SUPERVISION[i]["rationale"])
    (OOD_RUN_DIR / "resultado.json").write_text(json.dumps(ood_result, ensure_ascii=False, indent=2, allow_nan=False) + "\n")
    export_ood = OOD_RUN_DIR / "avaliacao-fora-dominio.zip"
    with zipfile.ZipFile(export_ood, "w", zipfile.ZIP_DEFLATED) as archive:
        for name in ("inputs.jsonl", "supervisao.jsonl", "protocolo.json"):
            archive.write(GITHUB_HEADS_DIR.parent / name, name)
        for name in ("previsoes.jsonl", "resultado.json"):
            archive.write(OOD_RUN_DIR / name, name)
    print("Avaliação salva em:", OOD_RUN_DIR.resolve())
    print("ZIP da avaliação:", export_ood.resolve())
